# Baseline MLE-STAR × MLE-bench Lite (backbone pareado)

Este notebook roda o **MLE-STAR oficial** (implementação open-source do Google em
[google/adk-samples](https://github.com/google/adk-samples), sample `machine-learning-engineering`)
nas **22 competições do MLE-bench Lite**, na **mesma infra Colab** e com o **mesmo backbone**
(`gemini-3-flash-preview` por padrão) usado pelo Kaggle Agents.

**Por que isso existe:** a comparação da tese usa números transplantados do paper do MLE-STAR
(Gemini 2.0 Flash / 2.5 Pro, cluster 8×V100, 3 seeds). Re-rodar o MLE-STAR com o mesmo modelo e
mesma máquina elimina o confound de backbone/infra e produz uma comparação de custo *medida* —
os dois pontos mais frágeis para publicação.

**Requisitos**
1. `GOOGLE_API_KEY` (Gemini Developer API) — em Colab, salve em *Secrets* (ícone de chave).
2. Credenciais do Kaggle (`kaggle.json` ou secrets `KAGGLE_USERNAME`/`KAGGLE_KEY`).
3. **Aceite as regras de cada competição no site do Kaggle** (uma vez por conta), senão o
   `mlebench prepare` falha com 403.
4. GPU (L4 recomendada) e disco: competições >10 GB podem estourar o disco do Colab
   (o MLE-STAR copia os dados para cada branch de solução). Veja `SKIP_LARGE_GB` abaixo.

**Saídas:** `RESULTS_DIR/results_mlestar.json` (por competição: medalhas, above_median, score bruto,
tempo de execução, seed) + tabela comparativa final no formato da Table 5 da monografia.

**Referências:** Nam et al., *MLE-STAR: Machine Learning Engineering Agent via Search and Targeted
Refinement* (NeurIPS 2025, arXiv:2506.15692); Chan et al., *MLE-bench* (arXiv:2410.07095).

In [ ]:
# 1) Dependências
# torch/pandas/sklearn já vêm no Colab; instalamos o ADK, o grader do MLE-bench e libs
# que as soluções geradas costumam importar.
!pip -q install "google-adk>=1.5.0" "google-genai>=1.9.0" python-dotenv kaggle lightgbm xgboost catboost
!pip -q install git+https://github.com/openai/mle-bench.git
!git clone --depth 1 https://github.com/google/adk-samples.git /content/adk-samples 2>/dev/null || echo 'adk-samples já clonado'

In [ ]:
# 2) Credenciais
import os
from pathlib import Path


def _load_secret(name: str) -> str | None:
    """Read a secret from Colab userdata or the environment."""
    try:
        from google.colab import userdata  # type: ignore

        value = userdata.get(name)
        if value:
            return value
    except Exception:
        pass
    return os.environ.get(name)


# Gemini Developer API (sem Vertex)
os.environ["GOOGLE_GENAI_USE_VERTEXAI"] = "0"
google_key = _load_secret("GOOGLE_API_KEY")
assert google_key, "Defina GOOGLE_API_KEY (Colab Secrets ou env)"
os.environ["GOOGLE_API_KEY"] = google_key

# Kaggle (para mlebench prepare)
kaggle_user = _load_secret("KAGGLE_USERNAME")
kaggle_key = _load_secret("KAGGLE_KEY")
if kaggle_user and kaggle_key:
    os.environ["KAGGLE_USERNAME"] = kaggle_user
    os.environ["KAGGLE_KEY"] = kaggle_key
    kaggle_dir = Path.home() / ".kaggle"
    kaggle_dir.mkdir(exist_ok=True)
    (kaggle_dir / "kaggle.json").write_text(
        f'{{"username": "{kaggle_user}", "key": "{kaggle_key}"}}'
    )
    os.chmod(kaggle_dir / "kaggle.json", 0o600)
assert (Path.home() / ".kaggle" / "kaggle.json").exists(), (
    "Coloque seu kaggle.json em ~/.kaggle/ ou defina KAGGLE_USERNAME/KAGGLE_KEY"
)
print("Credenciais OK")

In [ ]:
# 3) Configuração do experimento

# Backbone pareado com o Kaggle Agents (tese). Alternativas: gemini-2.5-flash / gemini-2.5-pro
MODEL = "gemini-3-flash-preview"

# Protocolo. O paper do MLE-STAR usa 3 seeds; 1 seed = paridade com a run única da tese.
SEEDS = [42, 43, 44]

# Orçamentos (o paper usa limite de 24h/competição; ajuste ao seu tempo de Colab)
MAX_WALL_CLOCK_S = 24 * 3600  # limite soft; subprocessos síncronos podem ultrapassá-lo
EXEC_TIMEOUT_S = 3600         # timeout por script gerado (config.exec_timeout)

# Forma do grafo do MLE-STAR (defaults do sample ADK; valores do paper em comentário)
NUM_SOLUTIONS = 2             # paper: 2
NUM_MODEL_CANDIDATES = 4      # paper: 4
OUTER_LOOP_ROUND = 4          # paper: 4
INNER_LOOP_ROUND = 4          # paper: 4
ENSEMBLE_LOOP_ROUND = 5       # paper: 5
USE_DATA_LEAKAGE_CHECKER = True
USE_DATA_USAGE_CHECKER = True

# Competições >SKIP_LARGE_GB são puladas (o MLE-STAR copia os dados por branch de solução;
# siim-isic ~116 GB não cabe no disco do Colab). None = tentar todas.
SKIP_LARGE_GB = None

# Diretórios (monte o Drive se quiser persistir resultados entre sessões)
RESULTS_DIR = Path("/content/mlestar_results")
WORK_ROOT = Path("/content/mlestar_work")
MLEBENCH_DATA_DIR = Path.home() / ".cache" / "mle-bench" / "data"
CLEANUP_AFTER_RUN = True      # apaga workspace + cache da competição após a nota (economiza disco)
FORCE_RERUN = False           # True re-executa competições já presentes no results_mlestar.json

RESULTS_DIR.mkdir(parents=True, exist_ok=True)
WORK_ROOT.mkdir(parents=True, exist_ok=True)

# MLE-bench Lite: 22 competições (task_type/lower alimentam os prompts do MLE-STAR)
COMPETITIONS = [
    {"id": "aerial-cactus-identification",                     "task_type": "Image Classification", "metric": "auc",                       "lower": False, "size_gb": 0.025},
    {"id": "aptos2019-blindness-detection",                    "task_type": "Image Classification", "metric": "quadratic_weighted_kappa",  "lower": False, "size_gb": 10.22},
    {"id": "dog-breed-identification",                         "task_type": "Image Classification", "metric": "log_loss",                  "lower": True,  "size_gb": 0.75},
    {"id": "dogs-vs-cats-redux-kernels-edition",               "task_type": "Image Classification", "metric": "log_loss",                  "lower": True,  "size_gb": 0.85},
    {"id": "histopathologic-cancer-detection",                 "task_type": "Image Classification", "metric": "auc",                       "lower": False, "size_gb": 7.76},
    {"id": "leaf-classification",                              "task_type": "Image Classification", "metric": "log_loss",                  "lower": True,  "size_gb": 0.036},
    {"id": "plant-pathology-2020-fgvc7",                       "task_type": "Image Classification", "metric": "auc",                       "lower": False, "size_gb": 0.8},
    {"id": "ranzcr-clip-catheter-line-classification",         "task_type": "Image Classification", "metric": "auc",                       "lower": False, "size_gb": 13.13},
    {"id": "siim-isic-melanoma-classification",                "task_type": "Image Classification", "metric": "auc",                       "lower": False, "size_gb": 116.16},
    {"id": "denoising-dirty-documents",                        "task_type": "Image to Image",       "metric": "rmse",                      "lower": True,  "size_gb": 0.06},
    {"id": "detecting-insults-in-social-commentary",           "task_type": "Text Classification",  "metric": "auc",                       "lower": False, "size_gb": 0.002},
    {"id": "jigsaw-toxic-comment-classification-challenge",    "task_type": "Text Classification",  "metric": "auc",                       "lower": False, "size_gb": 0.06},
    {"id": "random-acts-of-pizza",                             "task_type": "Text Classification",  "metric": "auc",                       "lower": False, "size_gb": 0.003},
    {"id": "spooky-author-identification",                     "task_type": "Text Classification",  "metric": "log_loss",                  "lower": True,  "size_gb": 0.002},
    {"id": "new-york-city-taxi-fare-prediction",               "task_type": "Tabular Regression",   "metric": "rmse",                      "lower": True,  "size_gb": 5.7},
    {"id": "nomad2018-predict-transparent-conductors",         "task_type": "Tabular Regression",   "metric": "rmsle",                     "lower": True,  "size_gb": 0.006},
    {"id": "tabular-playground-series-dec-2021",               "task_type": "Tabular Classification", "metric": "accuracy",                "lower": False, "size_gb": 0.7},
    {"id": "tabular-playground-series-may-2022",               "task_type": "Tabular Classification", "metric": "auc",                     "lower": False, "size_gb": 0.57},
    {"id": "mlsp-2013-birds",                                  "task_type": "Audio Classification", "metric": "auc",                       "lower": False, "size_gb": 0.585},
    {"id": "the-icml-2013-whale-challenge-right-whale-redux",  "task_type": "Audio Classification", "metric": "auc",                       "lower": False, "size_gb": 0.29},
    {"id": "text-normalization-challenge-english-language",    "task_type": "Sequence to Sequence", "metric": "accuracy",                  "lower": False, "size_gb": 0.01},
    {"id": "text-normalization-challenge-russian-language",    "task_type": "Sequence to Sequence", "metric": "accuracy",                  "lower": False, "size_gb": 0.01},
]
print(f"{len(COMPETITIONS)} competições configuradas | modelo: {MODEL} | seeds: {SEEDS}")

In [ ]:
# 4) Importar o MLE-STAR (ORDEM IMPORTA)
# O grafo de agentes é construído no IMPORT a partir de config.CONFIG — tudo que muda a
# forma do grafo (num_solutions, loops, checkers, modelo) precisa ser definido ANTES de
# importar machine_learning_engineering.agent.
import glob
import sys

os.environ["ROOT_AGENT_MODEL"] = MODEL  # obrigatório: root agent não tem fallback

agent_py = glob.glob("/content/adk-samples/**/machine_learning_engineering/agent.py", recursive=True)
assert agent_py, "Sample machine-learning-engineering não encontrado no adk-samples"
SAMPLE_DIR = Path(agent_py[0]).parent.parent  # .../machine-learning-engineering
sys.path.insert(0, str(SAMPLE_DIR))
print(f"MLE-STAR sample: {SAMPLE_DIR}")

from machine_learning_engineering.shared_libraries import config as mle_config

TASKS_DIR = SAMPLE_DIR / "machine_learning_engineering" / "tasks"

mle_config.CONFIG.agent_model = MODEL
mle_config.CONFIG.num_solutions = NUM_SOLUTIONS
mle_config.CONFIG.num_model_candidates = NUM_MODEL_CANDIDATES
mle_config.CONFIG.outer_loop_round = OUTER_LOOP_ROUND
mle_config.CONFIG.inner_loop_round = INNER_LOOP_ROUND
mle_config.CONFIG.ensemble_loop_round = ENSEMBLE_LOOP_ROUND
mle_config.CONFIG.use_data_leakage_checker = USE_DATA_LEAKAGE_CHECKER
mle_config.CONFIG.use_data_usage_checker = USE_DATA_USAGE_CHECKER
mle_config.CONFIG.exec_timeout = EXEC_TIMEOUT_S
mle_config.CONFIG.data_dir = str(TASKS_DIR) + "/"

# Só agora o import do agente (constrói o grafo com a config acima)
from machine_learning_engineering.agent import root_agent  # noqa: E402

print("root_agent importado com", mle_config.CONFIG.num_solutions, "soluções paralelas")

In [ ]:
# 5) Helpers MLE-bench: preparar dados, montar task dir, grade
import json
import shutil
import subprocess


def ensure_prepared(comp_id: str) -> Path:
    """Roda `mlebench prepare -c <comp>` se necessário e retorna o dir public/."""
    public = MLEBENCH_DATA_DIR / comp_id / "prepared" / "public"
    if public.exists() and any(public.iterdir()):
        return public
    print(f"  mlebench prepare -c {comp_id} (pode demorar: download do Kaggle)")
    proc = subprocess.run(
        ["mlebench", "prepare", "-c", comp_id],
        capture_output=True, text=True, timeout=4 * 3600,
    )
    if not public.exists():
        raise RuntimeError(
            f"prepare falhou para {comp_id}:\n{proc.stdout[-1000:]}\n{proc.stderr[-1000:]}"
        )
    return public


def get_description(comp_id: str) -> str:
    """Descrição oficial da competição via registry do mlebench (com fallback)."""
    try:
        from mlebench.registry import registry

        comp = registry.set_data_dir(MLEBENCH_DATA_DIR).get_competition(comp_id)
        desc = getattr(comp, "description", "") or ""
        if desc:
            return desc
    except Exception as e:
        print(f"  registry description indisponível ({e}); usando fallback")
    for md in (MLEBENCH_DATA_DIR / comp_id).rglob("description*.md"):
        return md.read_text(encoding="utf-8", errors="ignore")
    return ""


def build_task_dir(comp: dict) -> Path:
    """Monta tasks/<comp>/ no layout do MLE-STAR: task_description.txt + dados públicos.

    O create_workspace do MLE-STAR copia TODO arquivo do task dir (exceto nomes contendo
    'answer'), então colocamos apenas o public/ do mlebench — o private/ (gabarito) fica fora.
    """
    comp_id = comp["id"]
    public = ensure_prepared(comp_id)
    task_dir = TASKS_DIR / comp_id
    if task_dir.exists():
        shutil.rmtree(task_dir)
    shutil.copytree(public, task_dir)

    files_listing = "\n".join(sorted(p.name for p in task_dir.iterdir()))
    description = get_description(comp_id)
    task_description = (
        f"# Task ({comp['task_type']})\n\n"
        f"{description}\n\n"
        f"# Metric\n{comp['metric']}\n\n"
        f"# Submission\nProduce a submission file exactly in the format of "
        f"sample_submission.csv provided with the data.\n\n"
        f"# Available data files\n{files_listing}\n"
    )
    (task_dir / "task_description.txt").write_text(task_description, encoding="utf-8")
    return task_dir


def grade_submission(comp_id: str, submission_path: Path) -> dict:
    """`mlebench grade-sample` -> dict com valid_submission/score/medals/above_median."""
    try:
        proc = subprocess.run(
            ["mlebench", "grade-sample", str(submission_path), comp_id],
            capture_output=True, text=True, timeout=300,
        )
        output = proc.stdout + proc.stderr
        start, end = output.find("{"), output.rfind("}") + 1
        if start >= 0 and end > start:
            return json.loads(output[start:end])
        return {"valid_submission": False, "error": f"parse: {output[-400:]}"}
    except Exception as e:
        return {"valid_submission": False, "error": str(e)}


def cleanup_competition(comp_id: str, workspace_dir: Path) -> None:
    """Libera disco: task dir, workspace e cache do mlebench da competição."""
    for path in (TASKS_DIR / comp_id, workspace_dir, MLEBENCH_DATA_DIR / comp_id):
        shutil.rmtree(path, ignore_errors=True)


print("Helpers prontos")

In [ ]:
# 6) Runner: executa o pipeline MLE-STAR de ponta a ponta para uma (competição, seed)
import asyncio
import time

from google.adk.runners import InMemoryRunner
from google.genai import types


async def run_pipeline(comp_id: str) -> str:
    """Envia a instrução ao frontdoor agent e consome o stream de eventos do ADK."""
    runner = InMemoryRunner(agent=root_agent, app_name="mle-star-baseline")
    session = await runner.session_service.create_session(
        app_name=runner.app_name, user_id="baseline"
    )
    content = types.Content(
        parts=[types.Part(text=f"execute the {comp_id} task")], role="user"
    )
    last_text, n_events = "", 0
    async for event in runner.run_async(
        user_id=session.user_id, session_id=session.id, new_message=content
    ):
        n_events += 1
        author = getattr(event, "author", "?")
        if n_events % 10 == 0:
            print(f"    [{time.strftime('%H:%M:%S')}] {n_events} eventos (último: {author})")
        try:
            if event.content and event.content.parts and event.content.parts[0].text:
                last_text = event.content.parts[0].text
        except Exception:
            pass
    return last_text


async def run_one(comp: dict, seed: int) -> dict:
    comp_id = comp["id"]
    workspace_dir = WORK_ROOT / f"seed{seed}"
    result = {
        "competition_id": comp_id,
        "seed": seed,
        "model": MODEL,
        "task_type": comp["task_type"],
        "size_gb": comp["size_gb"],
        "valid_submission": False,
        "score": None,
        "gold_medal": False,
        "silver_medal": False,
        "bronze_medal": False,
        "any_medal": False,
        "above_median": False,
        "execution_time": 0.0,
        "error": None,
    }
    start = time.time()
    try:
        build_task_dir(comp)

        # Config runtime (lida pelo prepare_task a cada execução)
        mle_config.CONFIG.task_name = comp_id
        mle_config.CONFIG.task_type = comp["task_type"]
        mle_config.CONFIG.lower = comp["lower"]
        mle_config.CONFIG.seed = seed
        mle_config.CONFIG.workspace_dir = str(workspace_dir) + "/"

        try:
            await asyncio.wait_for(run_pipeline(comp_id), timeout=MAX_WALL_CLOCK_S)
        except asyncio.TimeoutError:
            result["error"] = f"wall-clock timeout ({MAX_WALL_CLOCK_S}s)"
            print(f"    TIMEOUT após {MAX_WALL_CLOCK_S}s — tentando grade parcial")

        # Submissão final do MLE-STAR: workspace/<task>/ensemble/final/submission.csv
        submission = workspace_dir / comp_id / "ensemble" / "final" / "submission.csv"
        if not submission.exists():
            candidates = list((workspace_dir / comp_id).rglob("submission.csv"))
            submission = candidates[0] if candidates else None

        if submission and submission.exists():
            saved = RESULTS_DIR / f"submission_{comp_id}_seed{seed}.csv"
            shutil.copy(submission, saved)
            grade = grade_submission(comp_id, submission)
            result.update(
                valid_submission=bool(grade.get("valid_submission")),
                score=grade.get("score"),
                gold_medal=bool(grade.get("gold_medal")),
                silver_medal=bool(grade.get("silver_medal")),
                bronze_medal=bool(grade.get("bronze_medal")),
                above_median=bool(grade.get("above_median")),
                grading_output=grade,
            )
            result["any_medal"] = bool(
                result["gold_medal"] or result["silver_medal"] or result["bronze_medal"]
            )
        else:
            result["error"] = result["error"] or "no submission produced"

        # Preserva o estado final (código gerado, scores) para o material suplementar
        final_state = workspace_dir / comp_id / "final_state.json"
        if final_state.exists():
            shutil.copy(final_state, RESULTS_DIR / f"final_state_{comp_id}_seed{seed}.json")
    except Exception as e:
        result["error"] = str(e)[:500]
    finally:
        result["execution_time"] = time.time() - start
        if CLEANUP_AFTER_RUN:
            cleanup_competition(comp_id, workspace_dir / comp_id)
    return result


print("Runner pronto")

In [ ]:
# 7) Loop principal (retomável: pula pares competição×seed já gravados)
results_path = RESULTS_DIR / "results_mlestar.json"
all_results = json.loads(results_path.read_text()) if results_path.exists() else []
done = {(r["competition_id"], r.get("seed", 42)) for r in all_results if not FORCE_RERUN}

for seed in SEEDS:
    for i, comp in enumerate(COMPETITIONS, 1):
        key = (comp["id"], seed)
        header = f"[{i}/{len(COMPETITIONS)}] {comp['id']} (seed={seed})"
        if key in done:
            print(f"{header} — já executado, pulando")
            continue
        if SKIP_LARGE_GB is not None and comp["size_gb"] > SKIP_LARGE_GB:
            print(f"{header} — PULADO ({comp['size_gb']} GB > {SKIP_LARGE_GB} GB)")
            all_results.append({
                "competition_id": comp["id"], "seed": seed, "model": MODEL,
                "task_type": comp["task_type"], "size_gb": comp["size_gb"],
                "skipped": True, "error": f"skipped: size > {SKIP_LARGE_GB} GB",
                "valid_submission": False, "any_medal": False, "above_median": False,
            })
            results_path.write_text(json.dumps(all_results, indent=2, default=str))
            continue
        print(f"\n{'=' * 70}\n{header}\n{'=' * 70}")
        result = await run_one(comp, seed)
        all_results.append(result)
        results_path.write_text(json.dumps(all_results, indent=2, default=str))
        medal = "🥇" if result["gold_medal"] else "🥈" if result["silver_medal"] else "🥉" if result["bronze_medal"] else "—"
        print(
            f"  -> valid={result['valid_submission']} score={result['score']} "
            f"medal={medal} above_median={result['above_median']} "
            f"({result['execution_time'] / 3600:.2f}h) erro={result['error']}"
        )

print(f"\nConcluído. Resultados em {results_path}")

In [ ]:
# 8) Agregação + tabela comparativa (formato da Table 5 da monografia)
import pandas as pd

df = pd.DataFrame(json.loads(results_path.read_text()))
skipped_mask = df.get("skipped", pd.Series(False, index=df.index)).fillna(False).astype(bool)
attempted = df[~skipped_mask]
n = len(attempted)


def pct(series) -> float:
    return 100.0 * series.fillna(False).astype(bool).sum() / max(n, 1)


mlestar_row = {
    "Sistema": f"MLE-STAR ({MODEL}, este notebook, n={n})",
    "Submissão válida %": round(pct(attempted["valid_submission"]), 1),
    "Acima da mediana %": round(pct(attempted["above_median"]), 1),
    "Medalhas %": round(pct(attempted["any_medal"]), 1),
    "Ouro %": round(pct(attempted.get("gold_medal", pd.Series(dtype=bool))), 1),
}

# Referências: tese (Kaggle Agents) e paper do MLE-STAR (Nam et al., 2025)
reference_rows = [
    {"Sistema": "AIDE (Gemini 2.0 Flash, paper)",        "Submissão válida %": 78.8,  "Acima da mediana %": 39.4, "Medalhas %": 25.8, "Ouro %": 12.1},
    {"Sistema": "MLE-STAR (Gemini 2.0 Flash, paper)",    "Submissão válida %": 95.5,  "Acima da mediana %": 63.6, "Medalhas %": 43.9, "Ouro %": 30.3},
    {"Sistema": "MLE-STAR (Gemini 2.5 Pro, paper)",      "Submissão válida %": 100.0, "Acima da mediana %": 83.3, "Medalhas %": 63.6, "Ouro %": 36.4},
    {"Sistema": "Kaggle Agents (Gemini 3.0 Flash, tese, protocolo legado)", "Submissão válida %": 100.0, "Acima da mediana %": 72.7, "Medalhas %": 59.1, "Ouro %": 27.3},
]

comparison = pd.DataFrame(reference_rows + [mlestar_row])
display(comparison)

print("\nPor competição:")
cols = ["competition_id", "seed", "valid_submission", "score", "any_medal", "above_median", "execution_time", "error"]
display(df[[c for c in cols if c in df.columns]])

total_h = attempted["execution_time"].fillna(0).sum() / 3600 if "execution_time" in attempted.columns else 0.0
print(f"\nTempo total de execução: {total_h:.1f} h GPU (custo L4 ~US$ {total_h * 0.67:.0f})")
comparison.to_csv(RESULTS_DIR / "comparison_table.csv", index=False)

## Notas metodológicas (para a seção de experimentos do paper)

1. **Backbone pareado:** este baseline usa o mesmo modelo do Kaggle Agents (`gemini-3-flash-preview`),
   eliminando o confound de geração de modelo da comparação original (paper: 2.0 Flash / 2.5 Pro).
2. **Mesma infra:** uma GPU de Colab, mesmo disco/CPU — o custo por competição passa a ser *medido*
   na mesma máquina, e não estimado por proxy de preço de V100.
3. **Seeds:** este notebook já usa `SEEDS = [42, 43, 44]`, como o protocolo de 3 seeds do
   MLE-STAR; reporte média, dispersão e teste pareado por competição.
4. **Loops:** `NUM_MODEL_CANDIDATES=4` e os loops 4/4/5 já estão configurados para a
   paridade planejada. Registre qualquer redução de orçamento antes de comparar resultados.
5. **Contaminação:** o MLE-STAR usa o `google_search` nativo do ADK para recuperar modelos —
   sem filtro anti-contaminação específico por competição (mesma condição do paper). O Kaggle
   Agents, após o patch de contaminação, filtra notebooks da própria competição; reporte ambos
   os protocolos de busca no paper.
6. **Competições puladas por disco** (`SKIP_LARGE_GB`) devem ser reportadas explicitamente na
   tabela (ex.: siim-isic ~116 GB não cabe no disco do Colab porque o MLE-STAR copia os dados
   por branch de solução).
7. **Limite de 24h:** `asyncio.wait_for` é um limite de orquestração, não um hard kill para
   ferramentas síncronas; registre o wall-clock real e qualquer excesso de até um timeout de componente.